# CIFAR-10 Quick Test - Google Colab (5 epochs)

Versi cepat untuk testing - hanya 5 epochs per model

⏱️ **Waktu eksekusi**: ~5-8 menit dengan GPU

In [ ]:
# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Config - QUICK TEST (5 epochs)
EPOCHS = 5
BATCH_SIZE = 32
NUM_CLASSES = 10
CLASSES = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Data loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✓ Data loaded - Train: {len(train_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# Simple VGG
class SimpleVGG(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Simple ResNet
class SimpleResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.layer1 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64)
        )
        
        self.pool = nn.MaxPool2d(2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x) + self.pool(x)
        x = self.avgpool(torch.relu(x))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

print("✓ Models defined")

In [ ]:
def train_model(model, name, epochs=5):
    print(f"\n🚀 Training {name}...")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    
    train_accs = []
    test_accs = []
    
    start_time = time.time()
    
    for epoch in range(epochs):
        # Train
        model.train()
        correct = 0
        total = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = 100 * correct / total
        train_accs.append(train_acc)
        
        # Test
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_acc = 100 * correct / total
        test_accs.append(test_acc)
        
        print(f"  Epoch {epoch+1}/{epochs}: Train={train_acc:.1f}%, Test={test_acc:.1f}%")
    
    elapsed = time.time() - start_time
    print(f"  ✓ Done! Time: {elapsed:.1f}s, Best Test Acc: {max(test_accs):.1f}%")
    
    return train_accs, test_accs, elapsed

# Train models
vgg = SimpleVGG(NUM_CLASSES)
resnet = SimpleResNet(NUM_CLASSES)

vgg_train, vgg_test, vgg_time = train_model(vgg, "VGG", EPOCHS)
resnet_train, resnet_test, resnet_time = train_model(resnet, "ResNet", EPOCHS)

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training accuracy
ax = axes[0]
ax.plot(vgg_train, label='VGG Train', marker='o')
ax.plot(vgg_test, label='VGG Test', marker='s')
ax.plot(resnet_train, label='ResNet Train', marker='o')
ax.plot(resnet_test, label='ResNet Test', marker='s')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)

# Comparison
ax = axes[1]
models = ['VGG', 'ResNet']
test_accs = [max(vgg_test), max(resnet_test)]
times = [vgg_time, resnet_time]

colors = ['#1f77b4', '#2ca02c']
bars = ax.bar(models, test_accs, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Best Test Accuracy (%)')
ax.set_title('Model Comparison')
ax.set_ylim([0, 100])

for bar, acc, t in zip(bars, test_accs, times):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{acc:.1f}%\n({t:.0f}s)', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('quick_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Quick test completed!")
print(f"\nResults:")
print(f"  VGG:    Best Test Acc = {max(vgg_test):.1f}%  |  Time = {vgg_time:.1f}s")
print(f"  ResNet: Best Test Acc = {max(resnet_test):.1f}%  |  Time = {resnet_time:.1f}s")